In [1]:
import sys
import os
import pandas as pd
import json

# 1. Dynamically set the project root directory path
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# Add the project root directory to the Python search path so that you can import src.
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"📂 Project Root set to: {project_root}")

# 2. Import the core AI module (Ollama version)
try:
    from src.rag_engine import JobRAGEngine
    from src.hybrid_logic import HybridPipeline
    print("✅ Successfully imported AI modules (Local Ollama Version).")
except ImportError as e:
    print(f"❌ Import Error: {e}")
    print("Please check if the __init__.py file exists in the src/ folder, and whether the dependency packages are installed.")

📂 Project Root set to: D:\1111\python\Code\LinkedinJobs_SWE_Analyze
✅ Successfully imported AI modules (Local Ollama Version).


In [2]:
# 3. Define file path
csv_filename = "unique_data.csv"
csv_path = os.path.join(project_root, 'data', csv_filename)
# Define the cache location of the local vector library
chroma_cache_dir = os.path.join(project_root, "chroma_db_local")

# Check if the file exists
if not os.path.exists(csv_path):
    print(f"❌ Error: Cannot find file {csv_path}")
    print("Please ensure that you have placed unique_data.csv in the data folder of your project.")
else:
    print(f"✅ Data file found: {csv_path}")

    # Simply read the first few lines to see the content.
    df = pd.read_csv(csv_path)
    print(f"📊 Loaded {len(df)} job postings.")
    print(f"Columns: {df.columns.tolist()}")

✅ Data file found: D:\1111\python\Code\LinkedinJobs_SWE_Analyze\data\unique_data.csv
📊 Loaded 618 job postings.
Columns: ['title', 'company', 'location', 'posted_before', 'application_number', 'detail', 'description', 'url', 'job_id']


In [3]:
print("\n--- 🚀 Demo 1: Hybrid Extraction (Rules + Local AI) ---")
print("Initializing Hybrid Pipeline (connecting to Ollama)...")

pipeline = HybridPipeline()

# --- Test A: Simulated Swedish text (to verify translation and rule-following capabilities) ---
sample_text_se = """
Vi söker en erfaren Lagerarbetare till Göteborg.
Krav: B-körkort och truckkort.
Skicka ansökan till jobb@lager.se.
Detta är en tjänst på distans ibland.
"""

print(f"\n📄 Processing Sample Swedish Text:\n{sample_text_se.strip()}...")
result_se = pipeline.process_job(sample_text_se)

print("\n✅ Extraction Result (JSON):")
print(json.dumps(result_se, indent=2, ensure_ascii=False))

# --- Test B: Using real data from a CSV file ---
if not df.empty:
    # Select the first description with content.
    real_desc = df['description'].dropna().iloc[0]
    print(f"\n📄 Processing Real Data from CSV (First 150 chars):\n{real_desc[:150]}...")

    result_real = pipeline.process_job(real_desc)

    print("\n✅ Real Data Result:")
    print(json.dumps(result_real, indent=2, ensure_ascii=False))


--- 🚀 Demo 1: Hybrid Extraction (Rules + Local AI) ---
Initializing Hybrid Pipeline (connecting to Ollama)...
Initializing Hybrid Pipeline (Rule Engine + AI)...

📄 Processing Sample Swedish Text:
Vi söker en erfaren Lagerarbetare till Göteborg.
Krav: B-körkort och truckkort.
Skicka ansökan till jobb@lager.se.
Detta är en tjänst på distans ibland....
⚠️ Extraction Error: Failed to parse JobPostingSchema from completion {"properties": {"normalized_title": {"title": "Job Title", "description": "The standardized job title in English.", "type": "string"}, "min_years_experience": {"title": "Min Years Experience", "description": "Minimum years of experience required. 0 if unknown.", "type": "integer"}, "hard_skills": {"title": "Hard Skills", "description": "List of technical skills/tools.", "type": "array", "items": {"type": "string"}}, "soft_skills": {"title": "Soft Skills", "description": "List of soft skills.", "type": "array", "items": {"type": "string"}}, "is_remote": {"title": "Is Remo

In [4]:
print("\n--- 🚀 Demo 2: Local RAG (Chat with your Data) ---")
print("Initializing RAG Engine with Local Embeddings...")
print("(If running for the first time, this may take 1-2 minutes to embed all data.)")

# Initialize the RAG engine
# source_column="description" Tell the program which column of the CSV to read.
rag = JobRAGEngine(
    csv_path=csv_path,
    source_column="description",
    persist_dir=chroma_cache_dir
)

# Define the question you want to ask.
question = "What are the common hard skills required for Service or Waiter roles?"
print(f"\n❓ Question: {question}")

# Execute query
# Local model inference speed depends on computer configuration and typically takes 10-30 seconds.
print("Thinking...")
response = rag.query(question)

print(f"\n🤖 Local AI Answer:\n{response['answer']}")
print(f"\n📚 Source Documents Used: {len(response['source_docs'])}")


--- 🚀 Demo 2: Local RAG (Chat with your Data) ---
Initializing RAG Engine with Local Embeddings...
(If running for the first time, this may take 1-2 minutes to embed all data.)
Loading documents from D:\1111\python\Code\LinkedinJobs_SWE_Analyze\data\unique_data.csv...
Initializing Local Embeddings...
Creating new vector store (Local)...
Initializing Local LLM...

❓ Question: What are the common hard skills required for Service or Waiter roles?
Thinking...
Thinking (Local LLM takes a bit longer)...

🤖 Local AI Answer:
Based on general knowledge and job postings, here are some common hard skills required for Service or Waiter roles:

1. **Communication skills**: Verbal and written communication to interact with customers, take orders, and provide service.
2. **Multitasking**: Ability to handle multiple tables, orders, and tasks simultaneously while maintaining attention to detail and providing excellent customer service.
3. **Time management**: Effective time management to ensure timely